# 12 — Held-out classifier evaluation

**Purpose:** Evaluate the selected checkpoint on fixed fold_2 only.

**Required inputs:** A non-smoke best_model.pth and saved subject manifests.

**Saved outputs:** outputs/classifier/paper_results/classification_metrics.*, plots and predictions.

## 1. Load central configuration

Set `cnmc_root` once in ignored `config.local.yaml`. Source images are never moved or changed.

In [ ]:
from pathlib import Path
import os, sys
search = [Path.cwd(), *Path.cwd().parents]
if os.environ.get("LEUKEMIA_YOLO_ROOT"):
    search.insert(0, Path(os.environ["LEUKEMIA_YOLO_ROOT"]))
for candidate in search:
    if (candidate / "config.example.yaml").is_file() and (candidate / "src" / "classifier_config.py").is_file():
        sys.path.insert(0, str(candidate)); break
from src.classifier_config import classifier_config, classifier_output
cfg = classifier_config()
out = classifier_output(cfg)
print("Project:", cfg["root"])
print("Config:", cfg["config_path"])
print("C-NMC source:", cfg["cnmc_root"])

## 2. Select exact checkpoint

Pin the run you want to report; never choose it using test results.

In [ ]:
from src.classifier_config import latest_classifier_run
checkpoint = latest_classifier_run(cfg) / "best_model.pth"
print("Selected checkpoint:", checkpoint)

## 3. Evaluate test fold only

Positive class 1 is ALL/Blast. The script saves checkpoint SHA-256 to prevent result mixing.

In [ ]:
from src.classifier_evaluation import evaluate
from src.cnmc_splits import create_or_load_splits
splits, summary = create_or_load_splits(cfg)
print("Test cells:", len(splits["test"]), "subjects:", splits["test"].subject_id.nunique())
metrics = evaluate(cfg, checkpoint)
display(__import__("pandas").DataFrame([metrics]).T)

## 4. Review diagnostic figures

Include false negatives in subsequent error analysis.

In [ ]:
from IPython.display import Image as DisplayImage, display
for name in ("confusion_matrix.png", "confusion_matrix_normalized.png", "roc_curve.png", "pr_curve.png", "evaluation_examples.png"):
    file = out / "paper_results" / name
    if file.is_file(): display(DisplayImage(filename=str(file)))
assert metrics["split"] == "fold_2" and metrics["positive_class"] == "Leukemic Blast (ALL)"
print("PASS — held-out cell-level metrics saved; not patient-level clinical performance")

## Next

Open **13_EfficientNet_B3_GradCAM.ipynb** to explain selected predictions.